# MIT516 — Data Analytics for Cyber Security
## Assessment D: Spam / Phishing Email Detection
### Dataset: UCI Spambase (spambase.data)
---
**Pipeline Steps**
1. Data Loading & Inspection
2. Data Preprocessing
3. Exploratory Data Analysis (EDA)
4. Anomaly / Threat Detection
5. Machine Learning Classifier
6. Model Evaluation
7. Security Visualisation (Dashboard)

---
## Step 1 — Data Loading & Inspection

In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

# ── Column names from UCI documentation ──────────────────────────────────────
word_freq_cols = [
    'word_freq_make', 'word_freq_address', 'word_freq_all', 'word_freq_3d',
    'word_freq_our', 'word_freq_over', 'word_freq_remove', 'word_freq_internet',
    'word_freq_order', 'word_freq_mail', 'word_freq_receive', 'word_freq_will',
    'word_freq_people', 'word_freq_report', 'word_freq_addresses',
    'word_freq_free', 'word_freq_business', 'word_freq_email', 'word_freq_you',
    'word_freq_credit', 'word_freq_your', 'word_freq_font', 'word_freq_000',
    'word_freq_money', 'word_freq_hp', 'word_freq_hpl', 'word_freq_george',
    'word_freq_650', 'word_freq_lab', 'word_freq_labs', 'word_freq_telnet',
    'word_freq_857', 'word_freq_data', 'word_freq_415', 'word_freq_85',
    'word_freq_technology', 'word_freq_1999', 'word_freq_parts',
    'word_freq_pm', 'word_freq_direct', 'word_freq_cs', 'word_freq_meeting',
    'word_freq_original', 'word_freq_project', 'word_freq_re',
    'word_freq_edu', 'word_freq_table', 'word_freq_conference'
]
char_freq_cols = [
    'char_freq_semicolon', 'char_freq_paren', 'char_freq_bracket',
    'char_freq_exclaim', 'char_freq_dollar', 'char_freq_hash'
]
capital_cols = [
    'capital_run_length_average', 'capital_run_length_longest',
    'capital_run_length_total'
]
all_cols = word_freq_cols + char_freq_cols + capital_cols + ['label']

# ── Load dataset ──────────────────────────────────────────────────────────────
df = pd.read_csv('spambase.data', header=None, names=all_cols)

print('=== Dataset Shape ===')
print(f'Rows: {df.shape[0]}, Columns: {df.shape[1]}')
print()
print('=== First 5 Rows ===')
df.head()

FileNotFoundError: [Errno 2] No such file or directory: 'spambase.data'

In [3]:
print('=== Data Types & Non-Null Counts ===')
df.info()

=== Data Types & Non-Null Counts ===


NameError: name 'df' is not defined

In [ ]:
print('=== Summary Statistics ===')
df.describe().round(4)

In [ ]:
print('=== Missing Values ===')
missing = df.isnull().sum()
print(f'Total missing values: {missing.sum()}')
print(missing[missing > 0] if missing.sum() > 0 else 'No missing values found.')

print()
print('=== Class Distribution ===')
class_counts = df['label'].value_counts()
class_pct    = df['label'].value_counts(normalize=True) * 100
class_df = pd.DataFrame({'Count': class_counts, 'Percentage (%)': class_pct.round(2)})
class_df.index = ['Spam (1)', 'Legitimate (0)']
print(class_df)

---
## Step 2 — Data Preprocessing

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from imblearn.over_sampling import SMOTE

# ── Separate features and target ──────────────────────────────────────────────
X = df.drop('label', axis=1)
y = df['label']

print(f'Feature matrix shape : {X.shape}')
print(f'Target vector shape  : {y.shape}')

In [ ]:
# ── Check for infinite values ─────────────────────────────────────────────────
inf_count = np.isinf(X).sum().sum()
print(f'Infinite values in features: {inf_count}')

# Replace any infinities with NaN then fill with column median
X = X.replace([np.inf, -np.inf], np.nan)
X = X.fillna(X.median())
print('Missing/infinite values handled.')

In [ ]:
# ── Train / Test Split (80/20, stratified) ────────────────────────────────────
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
print(f'Training set   : {X_train.shape[0]} samples')
print(f'Test set       : {X_test.shape[0]} samples')

In [ ]:
# ── Feature Scaling (StandardScaler) ─────────────────────────────────────────
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled  = scaler.transform(X_test)

X_train_scaled_df = pd.DataFrame(X_train_scaled, columns=X.columns)
print('Feature scaling complete.')
print('Scaled training data — first 3 rows:')
pd.DataFrame(X_train_scaled, columns=X.columns).head(3).round(3)

In [ ]:
# ── SMOTE — address class imbalance ──────────────────────────────────────────
print('Class distribution BEFORE SMOTE:')
print(pd.Series(y_train).value_counts().to_dict())

smote = SMOTE(random_state=42)
X_train_sm, y_train_sm = smote.fit_resample(X_train_scaled, y_train)

print('Class distribution AFTER SMOTE:')
print(pd.Series(y_train_sm).value_counts().to_dict())
print(f'Resampled training set size: {X_train_sm.shape[0]}')

---
## Step 3 — Exploratory Data Analysis (EDA)

In [ ]:
# ── 3a. Class distribution bar chart ─────────────────────────────────────────
fig, ax = plt.subplots(figsize=(6, 4))
labels  = ['Legitimate (0)', 'Spam (1)']
counts  = df['label'].value_counts().sort_index().values
colors  = ['#2196F3', '#F44336']
bars    = ax.bar(labels, counts, color=colors, edgecolor='black', width=0.5)
for bar, count in zip(bars, counts):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 30,
            str(count), ha='center', fontsize=11, fontweight='bold')
ax.set_title('Class Distribution — Spambase Dataset', fontsize=13, fontweight='bold')
ax.set_ylabel('Number of Emails')
ax.set_ylim(0, max(counts) * 1.15)
plt.tight_layout()
plt.savefig('fig1_class_distribution.png', dpi=150)
plt.show()
print('Figure 1 saved.')

In [ ]:
# ── 3b. Distribution of key spam-indicator features ───────────────────────────
key_features = [
    'word_freq_free', 'word_freq_money', 'word_freq_your',
    'char_freq_exclaim', 'char_freq_dollar', 'capital_run_length_average'
]

fig, axes = plt.subplots(2, 3, figsize=(14, 8))
axes = axes.flatten()

for i, feat in enumerate(key_features):
    for lbl, color, name in [(0, '#2196F3', 'Legitimate'), (1, '#F44336', 'Spam')]:
        subset = df[df['label'] == lbl][feat]
        axes[i].hist(subset, bins=40, alpha=0.6, color=color, label=name, density=True)
    axes[i].set_title(feat.replace('_', ' ').title(), fontsize=9)
    axes[i].set_xlabel('Frequency')
    axes[i].set_ylabel('Density')
    axes[i].legend(fontsize=8)

fig.suptitle('Figure 2: Feature Distributions — Spam vs Legitimate', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('fig2_feature_distributions.png', dpi=150)
plt.show()
print('Figure 2 saved.')

In [ ]:
# ── 3c. Correlation heatmap (top 20 features by variance) ─────────────────────
top_var_features = X.var().nlargest(20).index.tolist()
corr_matrix = df[top_var_features + ['label']].corr()

fig, ax = plt.subplots(figsize=(14, 11))
mask = np.triu(np.ones_like(corr_matrix, dtype=bool))
sns.heatmap(
    corr_matrix, mask=mask, annot=True, fmt='.2f',
    cmap='RdYlBu_r', center=0, linewidths=0.5,
    annot_kws={'size': 6}, ax=ax
)
ax.set_title('Figure 3: Correlation Heatmap — Top 20 Features + Label', fontsize=13, fontweight='bold')
plt.xticks(rotation=45, ha='right', fontsize=7)
plt.yticks(fontsize=7)
plt.tight_layout()
plt.savefig('fig3_correlation_heatmap.png', dpi=150)
plt.show()
print('Figure 3 saved.')

In [ ]:
# ── 3d. Top features correlated with spam label ───────────────────────────────
label_corr = df.corr()['label'].drop('label').abs().sort_values(ascending=False)
top15_corr = label_corr.head(15)

fig, ax = plt.subplots(figsize=(8, 6))
colors = ['#F44336' if c > 0 else '#2196F3' 
          for c in df.corr()['label'].drop('label').loc[top15_corr.index]]
ax.barh(top15_corr.index[::-1], top15_corr.values[::-1], color=colors[::-1], edgecolor='black')
ax.set_xlabel('|Pearson Correlation with Spam Label|')
ax.set_title('Figure 4: Top 15 Features Correlated with Spam', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('fig4_top_correlated_features.png', dpi=150)
plt.show()
print('Figure 4 saved.')

---
## Step 4 — Anomaly / Threat Detection

In [ ]:
from sklearn.ensemble import IsolationForest
from scipy.stats import zscore

# ── 4a. Z-score anomaly detection ─────────────────────────────────────────────
X_scaled_full = scaler.transform(X)  # scale full dataset
z_scores = np.abs(zscore(X_scaled_full, axis=0))
zscore_anomaly = (z_scores > 3).any(axis=1)  # flag rows with any z > 3

print('=== Z-score Anomaly Detection (threshold = 3) ===')
print(f'Total anomalies detected : {zscore_anomaly.sum()}')
print(f'Anomaly rate             : {zscore_anomaly.mean()*100:.2f}%')

# Check overlap with actual spam labels
z_df = pd.DataFrame({'label': y.values, 'z_anomaly': zscore_anomaly})
print()
print('Z-score anomalies vs actual label:')
print(pd.crosstab(z_df['label'], z_df['z_anomaly'],
                  rownames=['Actual Label'], colnames=['Z-score Anomaly']))

In [ ]:
# ── 4b. Isolation Forest ──────────────────────────────────────────────────────
iso_forest = IsolationForest(
    n_estimators=100,
    contamination=0.39,   # ~39% spam in dataset
    random_state=42,
    n_jobs=-1
)
iso_pred   = iso_forest.fit_predict(X_scaled_full)   # -1 = anomaly, 1 = normal
iso_scores = iso_forest.decision_function(X_scaled_full)  # negative = more anomalous

iso_anomaly = (iso_pred == -1).astype(int)

print('=== Isolation Forest Anomaly Detection ===')
print(f'Anomalies detected : {iso_anomaly.sum()}')
print(f'Normal flagged     : {(iso_anomaly == 0).sum()}')

# Overlap with true spam
iso_df = pd.DataFrame({'label': y.values, 'iso_anomaly': iso_anomaly})
print()
print('Isolation Forest anomalies vs actual label:')
print(pd.crosstab(iso_df['label'], iso_df['iso_anomaly'],
                  rownames=['Actual Label'], colnames=['IF Anomaly (1=anomalous)']))

In [ ]:
# ── 4c. Visualise Isolation Forest anomaly scores ────────────────────────────
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Anomaly score distribution
for lbl, color, name in [(0, '#2196F3', 'Legitimate'), (1, '#F44336', 'Spam')]:
    mask = (y.values == lbl)
    ax1.hist(iso_scores[mask], bins=50, alpha=0.65, color=color, label=name, density=True)
ax1.axvline(0, color='black', linestyle='--', linewidth=1.5, label='Decision boundary')
ax1.set_xlabel('Anomaly Score')
ax1.set_ylabel('Density')
ax1.set_title('Isolation Forest — Anomaly Score Distribution')
ax1.legend()

# Proportion of anomalies in spam vs legitimate
props = iso_df.groupby('label')['iso_anomaly'].mean() * 100
ax2.bar(['Legitimate (0)', 'Spam (1)'], props.values, color=['#2196F3', '#F44336'], edgecolor='black')
ax2.set_ylabel('% Flagged as Anomalous')
ax2.set_title('Isolation Forest — % Anomalies per Class')
for i, v in enumerate(props.values):
    ax2.text(i, v + 0.5, f'{v:.1f}%', ha='center', fontweight='bold')

fig.suptitle('Figure 5: Isolation Forest Anomaly Detection Results', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('fig5_isolation_forest.png', dpi=150)
plt.show()
print('Figure 5 saved.')

---
## Step 5 — Machine Learning Classifier

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.tree import DecisionTreeClassifier
from sklearn.model_selection import cross_val_score

# ── Train Random Forest (primary classifier) ──────────────────────────────────
rf_model = RandomForestClassifier(
    n_estimators=100,
    max_depth=None,
    min_samples_split=2,
    random_state=42,
    n_jobs=-1
)
rf_model.fit(X_train_sm, y_train_sm)
print('Random Forest trained on SMOTE-balanced data.')

# ── Cross-validation (5-fold) ─────────────────────────────────────────────────
cv_scores = cross_val_score(rf_model, X_train_sm, y_train_sm, cv=5, scoring='f1', n_jobs=-1)
print(f'5-Fold CV F1 Scores : {cv_scores.round(4)}')
print(f'Mean CV F1          : {cv_scores.mean():.4f} ± {cv_scores.std():.4f}')

In [ ]:
# ── Train additional classifiers for comparison ───────────────────────────────
dt_model = DecisionTreeClassifier(random_state=42, max_depth=10)
dt_model.fit(X_train_sm, y_train_sm)

nb_model = GaussianNB()
nb_model.fit(X_train_sm, y_train_sm)

print('Decision Tree and Naïve Bayes trained.')

---
## Step 6 — Model Evaluation

In [ ]:
from sklearn.metrics import (
    classification_report, confusion_matrix,
    roc_auc_score, RocCurveDisplay, ConfusionMatrixDisplay
)

# ── Predictions ───────────────────────────────────────────────────────────────
rf_pred  = rf_model.predict(X_test_scaled)
dt_pred  = dt_model.predict(X_test_scaled)
nb_pred  = nb_model.predict(X_test_scaled)

rf_proba = rf_model.predict_proba(X_test_scaled)[:, 1]
dt_proba = dt_model.predict_proba(X_test_scaled)[:, 1]
nb_proba = nb_model.predict_proba(X_test_scaled)[:, 1]

In [ ]:
# ── Classification Reports ────────────────────────────────────────────────────
for model_name, preds in [
    ('Random Forest', rf_pred),
    ('Decision Tree', dt_pred),
    ('Naïve Bayes',   nb_pred)
]:
    print(f'\n{'='*55}')
    print(f'  {model_name}')
    print(f'{'='*55}')
    print(classification_report(
        y_test, preds,
        target_names=['Legitimate (0)', 'Spam (1)']
    ))

In [ ]:
# ── Confusion Matrices ────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

for ax, preds, name, cmap in [
    (axes[0], rf_pred, 'Random Forest',  'Blues'),
    (axes[1], dt_pred, 'Decision Tree',  'Greens'),
    (axes[2], nb_pred, 'Naïve Bayes',    'Oranges')
]:
    cm = confusion_matrix(y_test, preds)
    disp = ConfusionMatrixDisplay(cm, display_labels=['Legitimate', 'Spam'])
    disp.plot(ax=ax, colorbar=False, cmap=cmap)
    ax.set_title(name, fontsize=11, fontweight='bold')

fig.suptitle('Figure 6: Confusion Matrices — All Classifiers', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('fig6_confusion_matrices.png', dpi=150)
plt.show()
print('Figure 6 saved.')

In [ ]:
# ── ROC Curves ────────────────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(8, 6))

for proba, name, color in [
    (rf_proba, 'Random Forest', '#F44336'),
    (dt_proba, 'Decision Tree', '#4CAF50'),
    (nb_proba, 'Naïve Bayes',   '#FF9800')
]:
    auc = roc_auc_score(y_test, proba)
    RocCurveDisplay.from_predictions(
        y_test, proba,
        name=f'{name} (AUC = {auc:.4f})',
        color=color, ax=ax
    )

ax.plot([0, 1], [0, 1], 'k--', linewidth=1, label='Random Classifier')
ax.set_title('Figure 7: ROC Curves — All Classifiers', fontsize=13, fontweight='bold')
ax.legend(loc='lower right')
ax.set_xlabel('False Positive Rate')
ax.set_ylabel('True Positive Rate')
plt.tight_layout()
plt.savefig('fig7_roc_curves.png', dpi=150)
plt.show()
print('Figure 7 saved.')

In [ ]:
# ── Model Performance Summary Table ──────────────────────────────────────────
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

rows = []
for name, preds, proba in [
    ('Random Forest', rf_pred, rf_proba),
    ('Decision Tree', dt_pred, dt_proba),
    ('Naïve Bayes',   nb_pred, nb_proba)
]:
    rows.append({
        'Model'     : name,
        'Accuracy'  : round(accuracy_score(y_test, preds), 4),
        'Precision' : round(precision_score(y_test, preds), 4),
        'Recall'    : round(recall_score(y_test, preds), 4),
        'F1-Score'  : round(f1_score(y_test, preds), 4),
        'AUC-ROC'   : round(roc_auc_score(y_test, proba), 4)
    })

summary_df = pd.DataFrame(rows).set_index('Model')
print('=== Model Performance Summary ===')
print(summary_df.to_string())
summary_df

---
## Step 7 — Security Visualisation Dashboard

In [ ]:
# ── Feature Importance (Random Forest) ───────────────────────────────────────
importances = pd.Series(rf_model.feature_importances_, index=X.columns)
top20_imp   = importances.nlargest(20).sort_values()

fig, ax = plt.subplots(figsize=(10, 7))
colors_imp = plt.cm.RdYlGn(np.linspace(0.2, 0.9, len(top20_imp)))
ax.barh(top20_imp.index, top20_imp.values, color=colors_imp, edgecolor='grey', linewidth=0.5)
ax.set_xlabel('Feature Importance (Gini)', fontsize=11)
ax.set_title('Figure 8: Top 20 Random Forest Feature Importances', fontsize=13, fontweight='bold')
ax.axvline(top20_imp.mean(), color='red', linestyle='--', linewidth=1.2, label=f'Mean importance')
ax.legend()
plt.tight_layout()
plt.savefig('fig8_feature_importance.png', dpi=150)
plt.show()
print('Figure 8 saved.')

In [ ]:
# ── Security Dashboard — comprehensive summary figure ─────────────────────────
fig = plt.figure(figsize=(18, 14))
fig.suptitle(
    'MIT516 — Spam / Phishing Email Detection\nSecurity Analytics Dashboard',
    fontsize=16, fontweight='bold', y=0.98
)

# -- Panel A: Class distribution (pie) ----------------------------------------
ax1 = fig.add_subplot(3, 3, 1)
spam_counts = df['label'].value_counts().sort_index()
ax1.pie(
    spam_counts.values,
    labels=['Legitimate', 'Spam'],
    colors=['#2196F3', '#F44336'],
    autopct='%1.1f%%',
    startangle=90,
    wedgeprops=dict(edgecolor='white', linewidth=2)
)
ax1.set_title('A. Class Distribution', fontweight='bold')

# -- Panel B: Model comparison bar chart --------------------------------------
ax2 = fig.add_subplot(3, 3, 2)
metrics_plot = summary_df[['Accuracy', 'Precision', 'Recall', 'F1-Score']]
x_pos = np.arange(len(metrics_plot.columns))
width = 0.25
for i, (model, row) in enumerate(metrics_plot.iterrows()):
    ax2.bar(x_pos + i * width, row.values, width=width, label=model, edgecolor='black', linewidth=0.5)
ax2.set_xticks(x_pos + width)
ax2.set_xticklabels(metrics_plot.columns, fontsize=8)
ax2.set_ylim(0.5, 1.05)
ax2.set_ylabel('Score')
ax2.set_title('B. Model Performance Comparison', fontweight='bold')
ax2.legend(fontsize=7)

# -- Panel C: AUC-ROC scores --------------------------------------------------
ax3 = fig.add_subplot(3, 3, 3)
auc_vals  = summary_df['AUC-ROC'].values
auc_names = summary_df.index.tolist()
bars3 = ax3.barh(auc_names, auc_vals,
                 color=['#F44336', '#4CAF50', '#FF9800'], edgecolor='black')
for bar, v in zip(bars3, auc_vals):
    ax3.text(v - 0.02, bar.get_y() + bar.get_height()/2,
             f'{v:.4f}', va='center', ha='right', color='white', fontweight='bold', fontsize=9)
ax3.set_xlim(0.5, 1.02)
ax3.set_xlabel('AUC-ROC')
ax3.set_title('C. AUC-ROC by Model', fontweight='bold')

# -- Panel D: Confusion Matrix (RF) -------------------------------------------
ax4 = fig.add_subplot(3, 3, 4)
cm_rf = confusion_matrix(y_test, rf_pred)
sns.heatmap(cm_rf, annot=True, fmt='d', cmap='Blues',
            xticklabels=['Legit', 'Spam'],
            yticklabels=['Legit', 'Spam'], ax=ax4,
            linewidths=1, linecolor='white')
ax4.set_title('D. Random Forest Confusion Matrix', fontweight='bold')
ax4.set_xlabel('Predicted')
ax4.set_ylabel('Actual')

# -- Panel E: Isolation Forest anomaly scores ---------------------------------
ax5 = fig.add_subplot(3, 3, 5)
for lbl, color, name in [(0, '#2196F3', 'Legitimate'), (1, '#F44336', 'Spam')]:
    mask = (y.values == lbl)
    ax5.hist(iso_scores[mask], bins=40, alpha=0.6, color=color, label=name, density=True)
ax5.axvline(0, color='black', linestyle='--', linewidth=1.5)
ax5.set_xlabel('Isolation Forest Score')
ax5.set_ylabel('Density')
ax5.set_title('E. Isolation Forest Anomaly Scores', fontweight='bold')
ax5.legend(fontsize=8)

# -- Panel F: Top 10 feature importances --------------------------------------
ax6 = fig.add_subplot(3, 3, 6)
top10 = importances.nlargest(10).sort_values()
colors6 = plt.cm.RdYlGn(np.linspace(0.2, 0.9, len(top10)))
ax6.barh(top10.index, top10.values, color=colors6, edgecolor='grey', linewidth=0.5)
ax6.set_xlabel('Importance')
ax6.set_title('F. Top 10 Feature Importances (RF)', fontweight='bold')

# -- Panel G: Mean word frequency spam vs legit --------------------------------
ax7 = fig.add_subplot(3, 1, 3)
top10_feats = importances.nlargest(10).index
spam_means  = df[df['label'] == 1][top10_feats].mean()
legit_means = df[df['label'] == 0][top10_feats].mean()
x_g = np.arange(len(top10_feats))
ax7.bar(x_g - 0.2, spam_means.values,  0.4, label='Spam',      color='#F44336', edgecolor='black')
ax7.bar(x_g + 0.2, legit_means.values, 0.4, label='Legitimate', color='#2196F3', edgecolor='black')
ax7.set_xticks(x_g)
ax7.set_xticklabels([f.replace('word_freq_', '').replace('char_freq_', '').replace('capital_run_length_', '')
                     for f in top10_feats], rotation=30, ha='right', fontsize=9)
ax7.set_ylabel('Mean Feature Value')
ax7.set_title('G. Mean Feature Values — Spam vs Legitimate (Top 10 Features)', fontweight='bold')
ax7.legend()

plt.tight_layout(rect=[0, 0, 1, 0.96])
plt.savefig('fig9_security_dashboard.png', dpi=150, bbox_inches='tight')
plt.show()
print('Figure 9 (Dashboard) saved.')

In [ ]:
# ── Final Summary ─────────────────────────────────────────────────────────────
print('=' * 60)
print('  FINAL RESULTS SUMMARY')
print('=' * 60)
print(f'Dataset       : UCI Spambase')
print(f'Total samples : {len(df)}')
print(f'Features      : {X.shape[1]}')
print(f'Spam emails   : {(y == 1).sum()} ({(y == 1).mean()*100:.1f}%)')
print(f'Legit emails  : {(y == 0).sum()} ({(y == 0).mean()*100:.1f}%)')
print()
print('--- Best Model: Random Forest ---')
best = summary_df.loc['Random Forest']
for metric, val in best.items():
    print(f'  {metric:<12}: {val:.4f}')
print()
print('--- Anomaly Detection (Isolation Forest) ---')
print(f'  Anomalies detected : {iso_anomaly.sum()} / {len(df)}')
print(f'  Anomaly rate       : {iso_anomaly.mean()*100:.2f}%')
print()
print('Notebook complete. All figures saved to working directory.')